# LLM Roundtable — Judge Run Analysis

Comparison of model outputs across the pipeline: primary responses, peer critiques,
refined responses, and the judge-selected response for each author.

**Metrics** — ROUGE-1, ROUGE-2, ROUGE-L (F-measure) for lexical overlap; **MiniLM** (`all-MiniLM-L6-v2`) embedding cosine for semantic similarity; lengths in words.

## Setup and Metric Definitions

In [1]:
from __future__ import annotations

import json
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

RUN_PATH = None          # None -> newest outputs/n8n/*/judge_run.json
OUTPUT_ROOT = Path("outputs/n8n")
SAVE_CSV = True
SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

run_path = Path(RUN_PATH) if RUN_PATH else max(
    OUTPUT_ROOT.glob("*/judge_run.json"), key=lambda p: p.stat().st_mtime)
run = json.loads(run_path.read_text(encoding="utf-8"))

models = run["models"]
primary = run["primary_responses"]
critiques = run["critiques"]      # critiques[i][j] = critique by j about i's primary
refined = run["refined_responses"]  # refined[i][j]  = i's response after j's critique
judgments = run["judgments"]
n = len(models)
short = [m.split("/")[-1] for m in models]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
embedder = SentenceTransformer(SEMANTIC_MODEL)
_embed_cache: dict[str, np.ndarray] = {}


def words(t):
    return len((t or "").split())


def _embed(text: str) -> np.ndarray | None:
    text = (text or "").strip()
    if not text:
        return None
    if text not in _embed_cache:
        _embed_cache[text] = embedder.encode(
            text, normalize_embeddings=True, show_progress_bar=False
        )
    return _embed_cache[text]


def semantic_cosine(a, b):
    """Cosine similarity of MiniLM sentence embeddings (L2-normalized)."""
    ea, eb = _embed(a), _embed(b)
    if ea is None or eb is None:
        return 0.0
    return float(np.dot(ea, eb))


def compare(a, b):
    """ROUGE F1 (lexical) + MiniLM embedding cosine (semantic)."""
    s = scorer.score(target=a or "", prediction=b or "")
    return {
        "rouge1": s["rouge1"].fmeasure,
        "rouge2": s["rouge2"].fmeasure,
        "rougeL": s["rougeL"].fmeasure,
        "semantic_cosine": semantic_cosine(a, b),
    }


print(run_path, "|", run["run_id"], "|", n, "models")
print("semantic model:", SEMANTIC_MODEL)

/Users/narenkhatwani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/narenkhatwani/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


outputs/n8n/b31df6c2/judge_run.json | b31df6c2 | 5 models
semantic model: sentence-transformers/all-MiniLM-L6-v2


## Judge Selection

`chosen[i]` — index of the critic whose refined version the judge selected for author `i`.

In [2]:
# Blind labels: "Version 5". Older runs used "LLM-5" or "expert 5".
LABEL_RE = re.compile(r"(?:LLM\s*-?\s*|expert\s+|version\s+)(\d+)", re.I)


def parse_selection(judgment, i):
    if not judgment or not judgment.strip():
        return None
    lines = [ln for ln in judgment.splitlines() if ln.strip()]
    for line in [ln for ln in lines if "select" in ln.lower()] + lines[:1] + [judgment]:
        m = LABEL_RE.search(line)
        if m:
            k = int(m.group(1)) - 1
            if 0 <= k < n and k != i and (refined[i][k] or "").strip():
                return k
            return None
    return None


chosen = [parse_selection(judgments[i], i) for i in range(n)]
for i in range(n):
    k = chosen[i]
    print(f"LLM-{i+1} {short[i]:<26} -> LLM-{k+1} {short[k]}" if k is not None
          else f"LLM-{i+1} {short[i]:<26} -> UNRESOLVED")

LLM-1 gpt-5.6-sol                -> LLM-5 claude-opus-4.8
LLM-2 kimi-k3                    -> LLM-3 qwen3.7-max
LLM-3 qwen3.7-max                -> LLM-1 gpt-5.6-sol
LLM-4 gemini-3.1-pro-preview     -> LLM-1 gpt-5.6-sol
LLM-5 claude-opus-4.8            -> LLM-1 gpt-5.6-sol


## 1. Response Length — Primary Responses

In [3]:
t1 = pd.DataFrame([
    {"llm": f"LLM-{i+1}", "model": short[i], "words": words(primary[i])}
    for i in range(n)
])
t1

,llm,model,words
0,LLM-1,gpt-5.6-sol,2642
1,LLM-2,kimi-k3,2585
2,LLM-3,qwen3.7-max,956
3,LLM-4,gemini-3.1-pro-preview,1057
4,LLM-5,claude-opus-4.8,1157


## 2. Pairwise Similarity — Primary Responses

In [4]:
t2 = pd.DataFrame([
    {"pair": f"LLM-{i+1} vs LLM-{j+1}", "model_a": short[i], "model_b": short[j],
     **compare(primary[i], primary[j])}
    for i, j in combinations(range(n), 2)
])
t2.round(3)

,pair,model_a,model_b,rouge1,rouge2,rougeL,semantic_cosine
0,LLM-1 vs LLM-2,gpt-5.6-sol,kimi-k3,0.697,0.232,0.209,0.802
1,LLM-1 vs LLM-3,gpt-5.6-sol,qwen3.7-max,0.395,0.114,0.142,0.677
2,LLM-1 vs LLM-4,gpt-5.6-sol,gemini-3.1-pro-preview,0.425,0.119,0.150,0.615
3,LLM-1 vs LLM-5,gpt-5.6-sol,claude-opus-4.8,0.474,0.131,0.158,0.754
4,LLM-2 vs LLM-3,kimi-k3,qwen3.7-max,0.405,0.112,0.147,0.635
5,LLM-2 vs LLM-4,kimi-k3,gemini-3.1-pro-preview,0.423,0.117,0.152,0.590
6,LLM-2 vs LLM-5,kimi-k3,claude-opus-4.8,0.470,0.118,0.163,0.685
7,LLM-3 vs LLM-4,qwen3.7-max,gemini-3.1-pro-preview,0.598,0.204,0.263,0.661
8,LLM-3 vs LLM-5,qwen3.7-max,claude-opus-4.8,0.543,0.120,0.202,0.711
9,LLM-4 vs LLM-5,gemini-3.1-pro-preview,claude-opus-4.8,0.508,0.130,0.178,0.601


## 3. Response Length — Critiques

In [5]:
t3 = pd.DataFrame([
    {"target": f"LLM-{i+1}", "target_model": short[i],
     "critic": f"LLM-{j+1}", "critic_model": short[j],
     "words": words(critiques[i][j])}
    for i in range(n) for j in range(n)
    if i != j and critiques[i][j] is not None
])
t3

,target,target_model,critic,critic_model,words
0,LLM-1,gpt-5.6-sol,LLM-2,kimi-k3,1182
1,LLM-1,gpt-5.6-sol,LLM-3,qwen3.7-max,706
2,LLM-1,gpt-5.6-sol,LLM-4,gemini-3.1-pro-preview,794
3,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,1010
4,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,1646
5,LLM-2,kimi-k3,LLM-3,qwen3.7-max,636
6,LLM-2,kimi-k3,LLM-4,gemini-3.1-pro-preview,651
7,LLM-2,kimi-k3,LLM-5,claude-opus-4.8,927
8,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,1601
9,LLM-3,qwen3.7-max,LLM-2,kimi-k3,1239


## 4. Pairwise Similarity — Critiques of the Same Primary Response

In [6]:
t4 = pd.DataFrame([
    {"target": f"LLM-{i+1}", "target_model": short[i],
     "pair": f"LLM-{j+1} vs LLM-{k+1}",
     **compare(critiques[i][j], critiques[i][k])}
    for i in range(n)
    for j, k in combinations([j for j in range(n) if j != i and critiques[i][j] is not None], 2)
])
t4.round(3)

,target,target_model,pair,rouge1,rouge2,rougeL,semantic_cosine
0,LLM-1,gpt-5.6-sol,LLM-2 vs LLM-3,0.467,0.094,0.137,0.591
1,LLM-1,gpt-5.6-sol,LLM-2 vs LLM-4,0.471,0.089,0.144,0.447
2,LLM-1,gpt-5.6-sol,LLM-2 vs LLM-5,0.565,0.119,0.164,0.542
3,LLM-1,gpt-5.6-sol,LLM-3 vs LLM-4,0.570,0.132,0.195,0.648
4,LLM-1,gpt-5.6-sol,LLM-3 vs LLM-5,0.422,0.076,0.139,0.650
5,LLM-1,gpt-5.6-sol,LLM-4 vs LLM-5,0.458,0.077,0.146,0.615
6,LLM-2,kimi-k3,LLM-1 vs LLM-3,0.383,0.059,0.118,0.756
7,LLM-2,kimi-k3,LLM-1 vs LLM-4,0.401,0.059,0.108,0.681
8,LLM-2,kimi-k3,LLM-1 vs LLM-5,0.468,0.084,0.122,0.659
9,LLM-2,kimi-k3,LLM-3 vs LLM-4,0.572,0.155,0.218,0.802


## 5. Response Length — Refined Responses

In [7]:
t5 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "critic": f"LLM-{j+1}", "critic_model": short[j],
     "words": words(refined[i][j])}
    for i in range(n) for j in range(n)
    if i != j and refined[i][j] is not None
])
t5

,author,author_model,critic,critic_model,words
0,LLM-1,gpt-5.6-sol,LLM-2,kimi-k3,3481
1,LLM-1,gpt-5.6-sol,LLM-3,qwen3.7-max,3100
2,LLM-1,gpt-5.6-sol,LLM-4,gemini-3.1-pro-preview,3251
3,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,3310
4,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,4746
5,LLM-2,kimi-k3,LLM-3,qwen3.7-max,3284
6,LLM-2,kimi-k3,LLM-4,gemini-3.1-pro-preview,2506
7,LLM-2,kimi-k3,LLM-5,claude-opus-4.8,2928
8,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,1582
9,LLM-3,qwen3.7-max,LLM-2,kimi-k3,1430


## 6. Pairwise Similarity — Refined Responses of the Same Author

In [8]:
t6 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "pair": f"after LLM-{j+1} vs after LLM-{k+1}",
     **compare(refined[i][j], refined[i][k])}
    for i in range(n)
    for j, k in combinations([j for j in range(n) if j != i and refined[i][j] is not None], 2)
])
t6.round(3)

,author,author_model,pair,rouge1,rouge2,rougeL,semantic_cosine
0,LLM-1,gpt-5.6-sol,after LLM-2 vs after LLM-3,0.713,0.229,0.240,0.685
1,LLM-1,gpt-5.6-sol,after LLM-2 vs after LLM-4,0.674,0.210,0.226,0.646
2,LLM-1,gpt-5.6-sol,after LLM-2 vs after LLM-5,0.690,0.229,0.222,0.654
3,LLM-1,gpt-5.6-sol,after LLM-3 vs after LLM-4,0.700,0.246,0.269,0.670
4,LLM-1,gpt-5.6-sol,after LLM-3 vs after LLM-5,0.688,0.236,0.237,0.657
5,LLM-1,gpt-5.6-sol,after LLM-4 vs after LLM-5,0.705,0.233,0.239,0.686
6,LLM-2,kimi-k3,after LLM-1 vs after LLM-3,0.569,0.150,0.151,0.466
7,LLM-2,kimi-k3,after LLM-1 vs after LLM-4,0.545,0.159,0.170,0.516
8,LLM-2,kimi-k3,after LLM-1 vs after LLM-5,0.557,0.131,0.138,0.508
9,LLM-2,kimi-k3,after LLM-3 vs after LLM-4,0.664,0.241,0.309,0.500


## 7. Similarity — Primary vs. Refined Responses

In [9]:
t7 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "critic": f"LLM-{j+1}", "critic_model": short[j],
     "primary_words": words(primary[i]), "refined_words": words(refined[i][j]),
     **compare(primary[i], refined[i][j])}
    for i in range(n) for j in range(n)
    if i != j and refined[i][j] is not None
])
t7.round(3)

,author,author_model,critic,critic_model,primary_words,refined_words,rouge1,rouge2,rougeL,semantic_cosine
0,LLM-1,gpt-5.6-sol,LLM-2,kimi-k3,2642,3481,0.687,0.268,0.264,0.771
1,LLM-1,gpt-5.6-sol,LLM-3,qwen3.7-max,2642,3100,0.707,0.316,0.361,0.693
2,LLM-1,gpt-5.6-sol,LLM-4,gemini-3.1-pro-preview,2642,3251,0.718,0.296,0.322,0.650
3,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,2642,3310,0.749,0.370,0.365,0.790
4,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,2585,4746,0.597,0.229,0.199,0.654
5,LLM-2,kimi-k3,LLM-3,qwen3.7-max,2585,3284,0.641,0.265,0.317,0.576
6,LLM-2,kimi-k3,LLM-4,gemini-3.1-pro-preview,2585,2506,0.692,0.316,0.394,0.511
7,LLM-2,kimi-k3,LLM-5,claude-opus-4.8,2585,2928,0.617,0.219,0.281,0.537
8,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,956,1582,0.555,0.228,0.227,0.641
9,LLM-3,qwen3.7-max,LLM-2,kimi-k3,956,1430,0.648,0.349,0.346,0.664


## 8. Response Length — Judge-Selected Responses

In [10]:
t8 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "chosen_critic": f"LLM-{chosen[i]+1}" if chosen[i] is not None else None,
     "chosen_critic_model": short[chosen[i]] if chosen[i] is not None else None,
     "words": words(refined[i][chosen[i]]) if chosen[i] is not None else None}
    for i in range(n)
])
t8

,author,author_model,chosen_critic,chosen_critic_model,words
0,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,3310
1,LLM-2,kimi-k3,LLM-3,qwen3.7-max,3284
2,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,1582
3,LLM-4,gemini-3.1-pro-preview,LLM-1,gpt-5.6-sol,1320
4,LLM-5,claude-opus-4.8,LLM-1,gpt-5.6-sol,2169


## 9. Pairwise Similarity — Judge-Selected Responses

In [11]:
chosen_text = [refined[i][chosen[i]] if chosen[i] is not None else None for i in range(n)]

t9 = pd.DataFrame([
    {"pair": f"LLM-{i+1} vs LLM-{j+1}", "model_a": short[i], "model_b": short[j],
     **compare(chosen_text[i], chosen_text[j])}
    for i, j in combinations(range(n), 2)
    if chosen_text[i] is not None and chosen_text[j] is not None
])
t9.round(3)

,pair,model_a,model_b,rouge1,rouge2,rougeL,semantic_cosine
0,LLM-1 vs LLM-2,gpt-5.6-sol,kimi-k3,0.564,0.118,0.146,0.492
1,LLM-1 vs LLM-3,gpt-5.6-sol,qwen3.7-max,0.473,0.105,0.138,0.719
2,LLM-1 vs LLM-4,gpt-5.6-sol,gemini-3.1-pro-preview,0.441,0.093,0.137,0.640
3,LLM-1 vs LLM-5,gpt-5.6-sol,claude-opus-4.8,0.514,0.084,0.120,0.609
4,LLM-2 vs LLM-3,kimi-k3,qwen3.7-max,0.450,0.099,0.139,0.526
5,LLM-2 vs LLM-4,kimi-k3,gemini-3.1-pro-preview,0.432,0.105,0.143,0.554
6,LLM-2 vs LLM-5,kimi-k3,claude-opus-4.8,0.523,0.080,0.124,0.497
7,LLM-3 vs LLM-4,qwen3.7-max,gemini-3.1-pro-preview,0.612,0.159,0.200,0.670
8,LLM-3 vs LLM-5,qwen3.7-max,claude-opus-4.8,0.545,0.093,0.149,0.685
9,LLM-4 vs LLM-5,gemini-3.1-pro-preview,claude-opus-4.8,0.499,0.099,0.141,0.689


## 10. Similarity — Primary vs. Judge-Selected Responses

## 11. Paper table — Primary vs. refined (judge selection)

One block per primary author (**LLM-P**): each peer refinement with length, ROUGE, **MiniLM cosine** (`Co. Sim`), and **X** when the judge selected that version. Suitable for copy into a paper (CSV + LaTeX below).

In [ ]:
from IPython.display import display

def paper_model_name(slug: str) -> str:
    return slug.split("/")[-1].replace("-preview", "")


paper_rows: list[dict] = []
for i in range(n):
    critics = [j for j in range(n) if j != i and refined[i][j] is not None]
    for idx, j in enumerate(critics):
        m = compare(primary[i], refined[i][j])
        paper_rows.append(
            {
                "LLM-P": paper_model_name(short[i]) if idx == 0 else "",
                "Refined by": paper_model_name(short[j]),
                "Length": words(refined[i][j]),
                "ROUGE-1": m["rouge1"],
                "ROUGE-2": m["rouge2"],
                "ROUGE-L": m["rougeL"],
                "Co. Sim": m["semantic_cosine"],
                "LLM-J selected": "X" if chosen[i] == j else "",
            }
        )

paper_table = pd.DataFrame(paper_rows)
paper_display = paper_table.copy()
for col in ("ROUGE-1", "ROUGE-2", "ROUGE-L", "Co. Sim"):
    paper_display[col] = paper_display[col].round(3)

display(
    paper_display.style.hide(axis="index").set_properties(
        **{"text-align": "center"},
        subset=["Length", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Co. Sim", "LLM-J selected"],
    )
)


def paper_table_to_latex(df: pd.DataFrame) -> str:
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{llrrrrrc}",
        r"\toprule",
        r"LLM-P & Refined by & Length & ROUGE-1 & ROUGE-2 & ROUGE-L & Co. Sim & LLM-J \\",
        r"\midrule",
    ]
    block_start = 0
    while block_start < len(df):
        block = []
        i = block_start
        while i < len(df):
            block.append(df.iloc[i])
            i += 1
            if i < len(df) and df.iloc[i]["LLM-P"]:
                break
        nrows = len(block)
        for ridx, row in enumerate(block):
            llm_p = row["LLM-P"] if ridx == 0 else ""
            if ridx == 0 and nrows > 1:
                llm_p = rf"\multirow{{{nrows}}}{{*}}{{{llm_p}}}"
            lines.append(
                " & ".join(
                    [
                        llm_p,
                        str(row["Refined by"]),
                        str(int(row["Length"])),
                        f"{row['ROUGE-1']:.3f}",
                        f"{row['ROUGE-2']:.3f}",
                        f"{row['ROUGE-L']:.3f}",
                        f"{row['Co. Sim']:.3f}",
                        str(row["LLM-J selected"]),
                    ]
                )
                + r" \\"
            )
        block_start = i
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\caption{Primary vs.\\ refined responses (MiniLM cosine).}", r"\label{tab:primary-refined}", r"\end{table}"])
    return "\n".join(lines)


paper_latex = paper_table_to_latex(paper_display)
print(paper_latex[:1200], "\n... (truncated; full string in paper_latex)")

## 12. Judge selection vs. metrics

For each author, rank the four peer refinements (table 7) on each metric. **Rank 1** = highest value. Shows whether the judge picked the metric leader or a different revision style (e.g. longer rewrite with lower ROUGE / Co. Sim).

In [ ]:
from IPython.display import Markdown, display

METRIC_COLS = [
    ("rouge1", "ROUGE-1"),
    ("rouge2", "ROUGE-2"),
    ("rougeL", "ROUGE-L"),
    ("semantic_cosine", "Co. Sim"),
    ("refined_words", "Length"),
]

selection_rows: list[dict] = []
for i in range(n):
    k = chosen[i]
    if k is None:
        continue
    sub = t7[t7["author"] == f"LLM-{i + 1}"].copy()
    chosen_label = f"LLM-{k + 1}"
    row: dict = {
        "author": f"LLM-{i + 1}",
        "author_model": paper_model_name(short[i]),
        "chosen_critic": chosen_label,
        "chosen_critic_model": paper_model_name(short[k]),
        "n_refinements": len(sub),
    }
    rank_ones = 0
    for col, label in METRIC_COLS:
        ranks = sub[col].rank(ascending=False, method="min")
        rank = int(ranks[sub["critic"] == chosen_label].iloc[0])
        row[f"rank_{label}"] = rank
        if rank == 1:
            rank_ones += 1
    row["metric_rank_1_wins"] = rank_ones
    selection_rows.append(row)

selection_analysis = pd.DataFrame(selection_rows)
display(selection_analysis.round(3))

summary_lines = [
    f"- Judge picked a **rank-1 ROUGE-1** refinement in "
    f"{(selection_analysis['rank_ROUGE-1'] == 1).sum()}/{len(selection_analysis)} authors.",
    f"- Judge picked a **rank-1 Co. Sim** refinement in "
    f"{(selection_analysis['rank_Co. Sim'] == 1).sum()}/{len(selection_analysis)} authors.",
    f"- Judge picked the **longest** refinement in "
    f"{(selection_analysis['rank_Length'] == 1).sum()}/{len(selection_analysis)} authors.",
    f"- Mean rank-1 metric wins for chosen refinement: "
    f"{selection_analysis['metric_rank_1_wins'].mean():.2f} / {len(METRIC_COLS)}.",
]
display(Markdown("\n".join(["**Summary**"] + summary_lines)))

In [12]:
t10 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "primary_words": words(primary[i]), "chosen_words": words(chosen_text[i]),
     **compare(primary[i], chosen_text[i])}
    for i in range(n) if chosen_text[i] is not None
])
t10.round(3)

,author,author_model,primary_words,chosen_words,rouge1,rouge2,rougeL,semantic_cosine
0,LLM-1,gpt-5.6-sol,2642,3310,0.749,0.370,0.365,0.790
1,LLM-2,kimi-k3,2585,3284,0.641,0.265,0.317,0.576
2,LLM-3,qwen3.7-max,956,1582,0.555,0.228,0.227,0.641
3,LLM-4,gemini-3.1-pro-preview,1057,1320,0.572,0.228,0.255,0.661
4,LLM-5,claude-opus-4.8,1157,2169,0.489,0.109,0.150,0.615


## Export

In [13]:
if SAVE_CSV:
    out = run_path.parent / "analysis"
    out.mkdir(exist_ok=True)
    tables = {
        "01_primary_length": t1,
        "02_primary_pairwise": t2,
        "03_critique_length": t3,
        "04_critique_pairwise": t4,
        "05_refined_length": t5,
        "06_refined_pairwise": t6,
        "07_primary_vs_refined": t7,
        "08_chosen_length": t8,
        "09_chosen_pairwise": t9,
        "10_primary_vs_chosen": t10,
        "11_paper_table": paper_table,
        "12_judge_selection_analysis": selection_analysis,
    }
    for name, df in tables.items():
        df.to_csv(out / f"{name}.csv", index=False)
        print(f"{name:<32} {len(df):>3} rows")
    (out / "11_paper_table.tex").write_text(paper_latex, encoding="utf-8")
    print("11_paper_table.tex")
    print("->", out)

01_primary_length          5 rows
02_primary_pairwise       10 rows
03_critique_length        20 rows
04_critique_pairwise      30 rows
05_refined_length         20 rows
06_refined_pairwise       30 rows
07_primary_vs_refined     20 rows
08_chosen_length           5 rows
09_chosen_pairwise        10 rows
10_primary_vs_chosen       5 rows
-> outputs/n8n/b31df6c2/analysis
